In [3]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd

In [4]:
BASE_URL="https://scrapeme.live/shop/"

def get_Results_Count(soup):
    results_count_string = soup.select_one(".woocommerce-result-count").get_text(strip=True) if soup.select_one(".woocommerce-result-count") else None
    first, last, total = [int(n) for n in re.findall(r'\d+', results_count_string)]
    return first, last, total

def get_Listing_Cards(soup):
    return soup.select(".product")

def parse_Listing_Card(card, page_number):
    product_link = card.select_one("a.woocommerce-LoopProduct-link.woocommerce-loop-product__link")["href"]
    product_title = card.select_one(".woocommerce-loop-product__title").get_text(strip=True) if card.select_one(".woocommerce-loop-product__title") else None
    product_price = card.select_one(".woocommerce-Price-amount.amount").get_text(strip=True) if card.select_one(".woocommerce-loop-product__title") else None

    item = {
        "pageNumber": page_number,
        "productLink": product_link,
        "productTitle": product_title,
        "productPrice": product_price
    }
    return item

In [5]:
def parse_product_meta(meta):
    if meta is None:
        return None, [], []

    sku_tag = meta.select_one(".sku")
    sku = sku_tag.get_text(strip=True) if sku_tag else None

    categories = [a.get_text(strip=True) for a in meta.select(".posted_in a")]
    tags = [a.get_text(strip=True) for a in meta.select(".tagged_as a")]

    return sku, categories, tags

def parse_Detail_Page(soup, product_url, page_number):
    product_title = soup.select_one("h1.product_title.entry-title").get_text(strip=True) if soup.select_one("h1.product_title.entry-title") else None
    product_price = soup.select_one(".price").get_text(strip=True) if soup.select_one(".price") else None
    product_summary_description = soup.select_one(".woocommerce-product-details__short-description").get_text(strip=True) if soup.select_one(".woocommerce-product-details__short-description") else None
    stock_level = soup.select_one(".stock").get_text(strip=True) if soup.select_one(".stock") else None
    sku, categories, tags = parse_product_meta(soup.select_one(".product_meta"))
    description_panel = soup.select_one("#tab-description") if soup.select_one("#tab-description") else None
    full_description = description_panel.select_one("p").get_text(strip=True) if description_panel else None

    item_details = {
        "pageNumber": page_number,
        "productName": product_title,
        "productPrice": product_price,
        "productSummaryDesc": product_summary_description,
        "productDetailedDesc": full_description,
        "stock": stock_level,
        "sku": sku,
        "categories": categories,
        "tags": tags,
        "productURL": product_url
    }
    return item_details
    

In [6]:
session = requests.Session()

Product_List = []

first, last, total = 0, 1, 2
page_number = 1;
while(last != total):
    page_url = BASE_URL + f"page/{page_number}/"
    try:
        response = session.get(page_url)
    except requests.RequestException as e:
        print(f"[Warning] Request failed for {page_url}: {e}")
        page_number += 1
        continue
        
    soup = BeautifulSoup(response.text, "html.parser")

    first, last, total = get_Results_Count(soup)

    print(f"Quering Items {first} - {last} from Page {page_number}")
    cards = get_Listing_Cards(soup)
    for card in cards:
        product_item = parse_Listing_Card(card, page_number)
        Product_List.append(product_item)

    page_number += 1
    time.sleep(0.2)
session.close()

Quering Items 1 - 16 from Page 1
Quering Items 17 - 32 from Page 2
Quering Items 33 - 48 from Page 3
Quering Items 49 - 64 from Page 4
Quering Items 65 - 80 from Page 5
Quering Items 81 - 96 from Page 6
Quering Items 97 - 112 from Page 7
Quering Items 113 - 128 from Page 8
Quering Items 129 - 144 from Page 9
Quering Items 145 - 160 from Page 10
Quering Items 161 - 176 from Page 11
Quering Items 177 - 192 from Page 12
Quering Items 193 - 208 from Page 13
Quering Items 209 - 224 from Page 14
Quering Items 225 - 240 from Page 15
Quering Items 241 - 256 from Page 16
Quering Items 257 - 272 from Page 17
Quering Items 273 - 288 from Page 18
Quering Items 289 - 304 from Page 19
Quering Items 305 - 320 from Page 20
Quering Items 321 - 336 from Page 21
Quering Items 337 - 352 from Page 22
Quering Items 353 - 368 from Page 23
Quering Items 369 - 384 from Page 24
Quering Items 385 - 400 from Page 25
Quering Items 401 - 416 from Page 26
Quering Items 417 - 432 from Page 27
Quering Items 433 - 448 

In [7]:
session = requests.Session()


All_Products = []
for product in Product_List:
    try:
        response = session.get(product["productLink"])
    except requests.RequestException as e:
        print(f"[Warning] Request failed for {product["productLink"]}: {e}")
        continue
        
    soup = BeautifulSoup(response.text, "html.parser")

    print(f"Parsing Data from {product["productLink"]}")
    product_full = parse_Detail_Page(soup, product["productLink"], product["pageNumber"])

    All_Products.append(product_full)
    time.sleep(0.2)
session.close()    

Parsing Data from https://scrapeme.live/shop/Bulbasaur/
Parsing Data from https://scrapeme.live/shop/Ivysaur/
Parsing Data from https://scrapeme.live/shop/Venusaur/
Parsing Data from https://scrapeme.live/shop/Charmander/
Parsing Data from https://scrapeme.live/shop/Charmeleon/
Parsing Data from https://scrapeme.live/shop/Charizard/
Parsing Data from https://scrapeme.live/shop/Squirtle/
Parsing Data from https://scrapeme.live/shop/Wartortle/
Parsing Data from https://scrapeme.live/shop/Blastoise/
Parsing Data from https://scrapeme.live/shop/Caterpie/
Parsing Data from https://scrapeme.live/shop/Metapod/
Parsing Data from https://scrapeme.live/shop/Butterfree/
Parsing Data from https://scrapeme.live/shop/Weedle/
Parsing Data from https://scrapeme.live/shop/Kakuna/
Parsing Data from https://scrapeme.live/shop/Beedrill/
Parsing Data from https://scrapeme.live/shop/Pidgey/
Parsing Data from https://scrapeme.live/shop/Pidgeotto/
Parsing Data from https://scrapeme.live/shop/Pidgeot/
Parsing 

In [12]:
before = len(All_Products)
if before > total:
    df_final = pd.DataFrame(All_Products).drop_duplicates(subset="productURL")
elif before < total:
    print("[Data Check] Missing Entries")
else:
    print("[Data Check] Successfull -- No Duplicate / Missing Entries")
after = before

[Data Check] Successfull -- No Duplicate / Missing Entries


In [9]:
df_final = pd.DataFrame(All_Products)
df_final["categories"] = df_final["categories"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
df_final["tags"] = df_final["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
df_final.to_csv("23L_0662_versionC_static_products.csv", index=False, encoding="utf-8")

# Scraping Methodology (Static Scraping)
- I used the developer tools on mozilla firefox. There is the option of selecting an element and displaying its code
- In the code I presume. I simply created the urls for each page and queried that url /page/{n}
- Total number of pages can be calculated using the total and current information in the listings page. I simply checked if the last on the current listing is equal to the total
- I had some problems with the class names. I did not know how to handle the spaces in between the classes (.). Also I had problems with getting the detailed summary. I was trying to query it using classes but it was not working so i used id instead
- Visual Check for correctness. And cross check for Counts

# Scraping Statistics

In [13]:
print(f"Total Pages: {page_number}")
print(f"Total Number of Records: {before}")
print(f"Total Number of Records Extracted: {before}")
print(f"Number of Duplicate Records: {before - after}")
print(f"Number of Missing Records: {after - before}")

Total Pages: 49
Total Number of Records: 755
Total Number of Records Extracted: 755
Number of Duplicate Records: 0
Number of Missing Records: 0
